# PharmaML — The Complete Docking Workflow (Zero to One Hundred)

One notebook, every stage of a molecular-docking campaign, in the order you
would actually run it:

| § | Stage | Toolkit |
|---|---|---|
| 1 | **Import the ligand library** — CSV in, validated library out | `pharmaml.docking` |
| 2 | **Import & prepare the protein** — PDB in, receptor PDBQT out | `pharmaml.docking` |
| 3 | Ligand preparation — SMILES → 3D PDBQT | `pharmaml.docking` |
| 4 | Grid box & GPF — enclose the binding site | `pharmaml.docking` |
| 5 | Run the engines — AutoDock4 / Vina (guarded) | `pharmaml.docking` |
| 6 | Parse AutoDock4 results — clusters, audit, campaign scan | `pharmaml.docking` |
| 7 | Parse Vina output — pose tables | `pharmaml.docking` |
| 8 | Rescoring — ligand efficiency + consensus | `pharmaml.docking` |
| 9 | Visualization — energy distribution, top ligands | `pharmaml.viz` |
| 10 | Publish — one-click HTML report + csv/md exports | `pharmaml.viz` |

**Setup** — the parsing/analysis path needs only the core package; ligand
**and** protein preparation additionally need RDKit + Meeko:

```bash
pip install -e ".[reports]"        # core + plotting
pip install rdkit meeko gemmi       # §1–§3 preparation
```

The demo campaign below is *synthetic but structurally identical* to real
AutoDock4 output, so every cell runs without a docking engine installed.
Point `DOCKING_DIR` at **your** campaign folder and re-run — the parsers,
rescoring and plots work unchanged on real `.dlg` trees.


> **No-output variant.** This notebook is the bandwidth-friendly twin of
> `complete_docking_workflow.ipynb`: identical cells, every output and inline figure stripped
> (30% of the original size). Open the full twin to read the
> pre-computed results; run this one top-to-bottom (`Shift+Enter`) to
> produce them yourself — the setup cell lists the optional dependencies
> each stage needs.


In [ ]:
import tempfile
import warnings
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd

from pharmaml import __version__

pd.set_option("display.max_columns", 14)
pd.set_option("display.width", 120)

# Silence the fork() DeprecationWarning the process pool emits on this
# Python build (harmless here, noisy in committed outputs).
warnings.filterwarnings("ignore", message=".*use of fork.*")

# Scratch directory for this run's artifacts (use any path you like).
WORK = Path(tempfile.mkdtemp(prefix="docking_workflow_"))

# RDKit's C++ parser prints errors for invalid SMILES; silence the noise
# (the pharmaml layer reports invalid rows explicitly).
try:
    from rdkit import RDLogger

    RDLogger.DisableLog("rdApp.*")
except ImportError:
    pass

print(f"pharmaml {__version__}")
print(f"artifacts -> {WORK}")

## 1 · Import the ligand library — CSV in

The legacy workflow started from a `ligands.csv` with a hard-coded
`SMILES` column — three scripts in the rar archive did this job with
three slightly different behaviours (`smilestopdbqt.py`, `smiqt.py`,
`ligand_preparation_autodock4.py`). `load_ligand_csv` unifies them:

* the SMILES column is **auto-detected** (`SMILES`, `smiles`,
  `canonical_smiles`, ...); the ligand-name column too, falling back
  to row numbers,
* every SMILES is **syntax-checked with RDKit** before any 3D
  embedding happens, so a typo costs seconds, not a crashed batch,
* invalid rows are reported explicitly and exported to
  `failed_smiles.csv` — the one good idea in
  `ligand_preparation_autodock4.py`, kept.

`prepare_ligand_library` then turns the valid rows into the campaign
tree every downstream tool expects: **one folder per ligand, each
containing a prepared `l.pdbqt`** — exactly the layout the original
scripts created, and what `scan_docking_directory` audits in §6.

In [ ]:
# A demo ligand library: 8 real drug-like molecules plus two
# deliberately broken rows, so the validation accounting has something
# to report. Swap this cell for your own CSV -- everything below runs
# unchanged.
LIGANDS_CSV = WORK / "ligands.csv"

ligand_table = pd.DataFrame(
    {
        "name": [
            "aspirin", "ibuprofen", "paracetamol", "caffeine",
            "naproxen", "benzocaine", "coumarin", "vanillin",
            "broken_row", "empty_row",
        ],
        "SMILES": [
            "CC(=O)Oc1ccccc1C(=O)O",
            "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
            "CC(=O)Nc1ccc(O)cc1",
            "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
            "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
            "CCOC(=O)c1ccc(N)cc1",
            "O=C1Oc2ccccc2C=C1",
            "COc1cc(C=O)ccc1O",
            "not_a_smiles",   # <- will be reported as invalid
            None,             # <- empty cell, also invalid
        ],
    }
)
ligand_table.to_csv(LIGANDS_CSV, index=False)
print(f"ligand library -> {LIGANDS_CSV}  ({len(ligand_table)} rows)")
ligand_table

In [ ]:
from pharmaml.docking import load_ligand_csv

library = load_ligand_csv(LIGANDS_CSV)
print(library)
library.frame

In [ ]:
# The invalid rows, with the reason -- and the failed_smiles.csv export
# the legacy script wrote after every batch:
if len(library.invalid_frame):
    print(library.invalid_frame[["name", "smiles", "error"]].to_string(index=False))
    failed_path = library.write_failures(WORK / "failed_smiles.csv")
    print(f"\ninvalid rows exported -> {failed_path.name}")
else:
    print("no invalid rows in this library")

In [ ]:
from pharmaml.docking import prepare_ligand_library

LIGAND_FOLDERS = WORK / "csv_ligand_prep"   # <- your campaign directory

prep = prepare_ligand_library(library, LIGAND_FOLDERS)
print(prep)
prep.frame[["folder", "name", "ok", "atoms", "torsdof"]]

In [ ]:
# The campaign tree (one folder per ligand, l.pdbqt inside) --
# scan_docking_directory-ready, and the layout the §6 demo campaign
# imitates:
for entry in sorted(LIGAND_FOLDERS.iterdir()):
    print(f"  {entry.relative_to(WORK)}" + ("/" if entry.is_dir() else ""))

## 2 · Import & prepare the protein — PDB in, receptor PDBQT out

The rar scripts assumed the hard part was already done: `gpfer.py`
hard-coded `receptor_types = "A C HD N NA OA SA"` and a ready-made
`r.pdbqt` in every ligand folder, prepared *outside* the codebase with
the retired AutoDockTools. `prepare_protein` brings that step into the
package:

* import a protein from a PDB file (or an already-prepared PDBQT
  receptor),
* clean it — waters, non-protein HETATM records (ions, buffers,
  glycans), alternate-location conformers, chain selection,
* write a rigid receptor PDBQT with **AutoDock4 atom types, Gasteiger
  partial charges and polar hydrogens** via Meeko's polymer pipeline,
* audit every atom that went in, came out, or was dropped.

Demo input: hen egg-white lysozyme (PDB entry
[1AKI](https://www.rcsb.org/1AKI), public domain) — *the* classic
docking-tutorial receptor — with its 78 crystallographic waters, plus a
sprinkling of the usual suspects a wet PDB contains (a chloride ion, a
sulfate, an alternate conformer) so every cleaning rule visibly fires.

`distribute_receptor` finally copies the prepared receptor into the §1
campaign folders — the job of the archive's `adder.py`, minus the
engine executables it also (needlessly) copied everywhere.

In [ ]:
import base64
import gzip

# Lysozyme 1AKI atom records, embedded gz+base64 so this notebook
# re-runs anywhere offline (also shipped as
# examples/data/lysozyme_1aki.pdb). In practice you write this file
# from your own source: an RCSB download, a crystallography
# collaborator, an AlphaFold export, ...
_LYSOZYME_BLOB = (
    "..."
)

PROTEIN_PDB = WORK / "lysozyme_wet.pdb"
LYSOZYME_PDB_TEXT = gzip.decompress(base64.b64decode(_LYSOZYME_BLOB)).decode()
PROTEIN_PDB.write_text(LYSOZYME_PDB_TEXT, encoding="utf-8")
n_atom = LYSOZYME_PDB_TEXT.count("ATOM")
n_het = LYSOZYME_PDB_TEXT.count("HETATM")
print(f"{PROTEIN_PDB.name}: {n_atom} ATOM + {n_het} HETATM records")

In [ ]:
from collections import Counter

atom_records = [l for l in PROTEIN_PDB.read_text().splitlines()
                if l.startswith(("ATOM", "HETATM"))]
hetatm_residues = Counter(
    l[17:20].strip() for l in atom_records if l.startswith("HETATM")
)
print(f"{sum(1 for l in atom_records if l.startswith('ATOM')):5d} ATOM   records  (the polymer)")
for res, count in sorted(hetatm_residues.items()):
    print(f"{count:5d} HETATM records  ({res})")

In [ ]:
def _hetatm(serial, name, resname, chain, resseq, x, y, z, element):
    return (
        f"HETATM{serial:5d} {name:>4s} {resname:>3s} {chain}{resseq:4d}    "
        f"{x:8.3f}{y:8.3f}{z:8.3f}{1.0:6.2f}{0.0:6.2f}          {element:>2s}"
    )

# 1AKI is unusually clean, so we sprinkle in what a typical wet PDB
# carries: a chloride counter-ion, a sulfate from the crystallization
# buffer, and an alternate conformer (altloc B) of an existing atom.
lines = PROTEIN_PDB.read_text().splitlines()
wet = list(lines)
wet.append(_hetatm(2001, "CL", "CL", "A", 301, 20.0, 20.0, 20.0, "Cl"))
for i, (nm, x, y, z) in enumerate(
    [("S", 21, 21, 21), ("O1", 22, 21, 21), ("O2", 20, 21, 21),
     ("O3", 21, 22, 21), ("O4", 21, 20, 21)]
):
    element = "S" if nm == "S" else "O"
    wet.append(_hetatm(2002 + i, nm, "SO4", "A", 302, float(x), float(y), float(z), element))
clone = list(lines[10])          # duplicate one atom as alternate conformer B
clone[16] = "B"
wet.insert(11, "".join(clone))
PROTEIN_PDB.write_text("\n".join(wet) + "\nTER\nEND\n", encoding="utf-8")
print(f"doctored input: {len(wet)} records (was {len(lines)})")

In [ ]:
from pharmaml.docking import prepare_protein

RECEPTOR = WORK / "r.pdbqt"     # the filename gpfer.py's GPF lines expect
protein = prepare_protein(PROTEIN_PDB, RECEPTOR)
print(protein)

In [ ]:
from pharmaml.docking import extract_pdbqt_atom_types

receptor_lines = RECEPTOR.read_text().splitlines()
print("\n".join(receptor_lines[:4]))
print(f"... ({len(receptor_lines)} lines total)\n")

# The seven types gpfer.py hard-coded ("A C HD N NA OA SA") -- now
# derived from the actual receptor instead of assumed:
print("receptor atom types:", extract_pdbqt_atom_types(RECEPTOR))
pd.Series(protein.atom_types).sort_values(ascending=False).to_frame("count")

In [ ]:
from pharmaml.viz import plot_receptor_atom_types

# The same census as a figure, colour-coded by chemistry — donor
# hydrogens, heteroatoms, non-polar carbons, metals/ions. A missing HD
# (donor) population is the classic symptom of a mis-prepared receptor:
receptor_fig = plot_receptor_atom_types(
    RECEPTOR, title="Lysozyme receptor — AutoDock atom types"
)

In [ ]:
from pharmaml.docking import distribute_receptor

targets = distribute_receptor(RECEPTOR, LIGAND_FOLDERS)
print(f"r.pdbqt distributed into {len(targets)} ligand folders of §1's campaign"
      "  (adder.py's job, without the engine binaries)")

In [ ]:
# Already have a prepared r.pdbqt from a collaborator or an older
# campaign? The same call re-filters it (waters, HETATM, altlocs)
# without any Meeko step -- charges and types pass through untouched:
passthrough = prepare_protein(RECEPTOR, WORK / "r_clean.pdbqt")
print(f"mode={passthrough.mode}, {passthrough.output_atoms} atoms, "
      f"charges kept: [{passthrough.charge_min:+.3f}, {passthrough.charge_max:+.3f}]")

## 3 · Ligand preparation — SMILES → 3D PDBQT

The legacy `smilestopdbqt.py` embedded paths and mutated global state;
`smiles_to_pdbqt` embeds each molecule with ETKDG, MMFF-optimises it, writes
the PDBQT through Meeko, and repairs any non-AD4 atom types — all with
explicit output directories.


In [ ]:
from pharmaml.docking import smiles_to_pdbqt, extract_pdbqt_atom_types

LIGAND_DIR = WORK / "ligands"

# Three demo ligands (in your project: loop over your screening library).
for name, smi in [
    ("aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
    ("ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ("caffeine", "Cn1cnc2c1c(=O)n(C)c(=O)n2C"),
]:
    path = smiles_to_pdbqt(smi, LIGAND_DIR, name=name)
    print(f"prepared {name:12s} -> {path.name}  ({path.stat().st_size} bytes)")

In [ ]:
# The atom types Meeko wrote — these drive the affinity maps in §4.
for pdbqt in sorted(LIGAND_DIR.glob("*.pdbqt")):
    print(f"{pdbqt.stem:12s}: {extract_pdbqt_atom_types(pdbqt)}")

In [ ]:
# What a prepared ligand looks like (torsion tree + partial charges).
print((LIGAND_DIR / "aspirin.pdbqt").read_text()[:400])

## 4 · Grid box, GPF & DPF — enclose the binding site

`compute_grid_box` derives the search box from a co-crystallised (reference)
ligand plus padding — the calculation `gpfer.py` did by hand. `write_gpf`
then emits one AutoDock4 grid parameter file with an affinity-map line per
ligand atom type (extracted from the PDBQT itself).
The `receptor` line points at `r.pdbqt` — the file §2 prepared and
`distribute_receptor` copied into each ligand folder.


In [ ]:
from pharmaml.docking import GridBox, compute_grid_box

# In a real project this is the CO-CRYSTALLISED ligand carved from the
# receptor PDB; here the aspirin we just prepared plays that role.
reference = LIGAND_DIR / "aspirin.pdbqt"

box = compute_grid_box(reference, padding=4.0)
print(f"centre : ({box.center_x:.2f}, {box.center_y:.2f}, {box.center_z:.2f}) A")
print(f"size   : {tuple(round(s, 1) for s in box.size)} A  "
      f"(npts {box.npts} @ {box.spacing} A spacing)")
print(f"vina   : {box.as_vina_kwargs()}")

In [ ]:
from pharmaml.docking import write_gpf

gpf = write_gpf(reference, receptor_file="r.pdbqt", box=box,
                output_path=WORK / "r.gpf")
print(gpf.read_text())

In [ ]:
from pharmaml.docking import write_dpf

# The DPF is AutoDock4's per-ligand instruction sheet (upgrades dpfer.py):
# ligand atom types, the "about" rotation centre — the ligand centroid —
# TORSDOF, and the full Lamarckian GA + Solis-Wets local-search block.
# Every GA knob is a keyword argument; ga_run defaults to 100.
dpf = write_dpf(reference, receptor_file="r.pdbqt", ga_run=100,
                output_path=WORK / "r.dpf")
for line in dpf.read_text().splitlines():
    if line.startswith(("ligand_types", "move ", "about ", "torsdof",
                        "ga_run", "set_ga", "set_psw1", "analysis")):
        print(line)

In [ ]:
from pharmaml.docking import check_torsion_consistency, scan_torsion_consistency

# Torsion QA (upgrades niggamatch.py / mismatch2.py): prepare_ligand4.py
# writes one active-torsion REMARK and one BRANCH record per rotatable
# bond, plus a final TORSDOF total — the ligand is only valid when they
# agree. The legacy scripts exported this audit to Excel with red fills;
# pharmaml returns a typed report and auto-detects the writer lineage
# (MGLTools three-way rule vs Meeko's TORSDOF == BRANCH invariant).
qa = check_torsion_consistency(reference)
print(f"reference : TORSDOF={qa.torsdof}  remarks={qa.active_remarks}  "
      f"branches={qa.branches}  ->  {qa.status()}")

# The MGLTools lineage demo — exactly what the rar scripts audited:
mgl_style = WORK / "mgl_style.pdbqt"
mgl_style.write_text(
    "REMARK  1  A    between atoms: T1_2-CA   T3_4-CB\n"
    "REMARK  2  A    between atoms: T5_6-CG   T7_8-CD\n"
    "ROOT\nATOM      1  C   LIG A   1       0.000   0.000   0.000  1.00  0.00     0.000 C\n"
    "ENDROOT\n"
    "BRANCH    1    2\nBRANCH    2    3\n"
    "TORSDOF 2\n",
    encoding="utf-8",
)
print("mgltools :", check_torsion_consistency(mgl_style).status(),
      "(all three counts agree)")
mgl_style.write_text(mgl_style.read_text().replace("TORSDOF 2", "TORSDOF 3"),
                     encoding="utf-8")
print("mgltools :", check_torsion_consistency(mgl_style).status(),
      "(TORSDOF lies — the mismatch the Excel report flagged red)")

torsion_frame = scan_torsion_consistency(LIGAND_DIR)
print(f"\nligands scanned: {len(torsion_frame)}")
print(torsion_frame.head(6).to_string(index=False))
assert (torsion_frame['status'] == 'OK').all(), 'torsion QA failed'

## 5 · Run the engines — AutoDock4 / Vina

`run_autodock4`, `run_vina` and `batch_run_directories` wrap the engines with
argument-list subprocess calls (no `shell=True`, no `os.chdir`), availability
checks *before* hours of queue time, and per-run timings.  Neither engine is
installed in this demo environment — the cell below shows the *actionable*
error you get instead of a cryptic crash:

> In a production environment (AutoDock Tools / Vina on `PATH`) the same
> calls return a `DockingRunResult` with timings and output paths.


In [ ]:
from pharmaml.docking import run_autodock4, run_vina
from pharmaml.exceptions import DockingError

# AutoDock4 needs autogrid4/autodock4 on PATH; Vina needs `vina`.
for label, call in [
    ("run_vina", lambda: run_vina(
        receptor="r.pdbqt", ligand=reference, out=WORK / "aspirin_out.pdbqt",
        center=(box.center_x, box.center_y, box.center_z),
        size=box.size, seed=42,
    )),
    ("run_autodock4", lambda: run_autodock4(WORK)),   # r.gpf exists -> binary check fires
]:
    try:
        call()
    except DockingError as exc:
        print(f"[{label}] {exc}")

## 6 · Parse AutoDock4 results — clusters, audit, campaign scan

The parsers accept **text or paths** and return typed dataclasses /
DataFrames, so they are unit-testable without a docking engine — and so this
notebook can build a demo campaign whose `.dlg` files are byte-compatible
with the real format:

* `parse_dlg_cluster_table` — one ligand's cluster table (the
  `readerwithdetails.py` regex, now typed);
* `parse_dlg_summary` — best (lowest-energy) *and* most-populated cluster;
* `check_docking_completion` — the `checker.py` audit: every folder needs
  its `r.dlg` **and** the `Successful Completion` sentinel;
* `scan_docking_directory` — the whole campaign in one table, with explicit
  status columns for failed folders instead of a crash.

The demo tree includes one folder with **no log at all** and one
**interrupted** run, so the audit columns are exercised.  Finally,
`batch_run_directories` re-runs the whole campaign *idempotently*: folders
whose `.dlg` already exists are skipped, and failures are collected as
results (not raised) — the difference between a queued night job that
finishes and one that dies at folder 3.


In [ ]:
# --- DEMO CAMPAIGN ------------------------------------------------------
# A synthetic AutoDock4 campaign: one folder per ligand, each with an
# r.dlg whose cluster table uses the exact real format.  One folder is
# missing its log entirely and one was interrupted before clustering, so
# the audit has something to find.  (In your project: skip this cell and
# set DOCKING_DIR to your real campaign folder — e.g. the tree §1–§2
# just built.)
def make_dlg(seed: int, n_clusters: int = 5, total_runs: int = 50) -> str:
    rng = np.random.default_rng(seed)
    lowest = np.sort(rng.uniform(-11.8, -4.5, size=n_clusters))[::-1]
    sizes = rng.multinomial(total_runs, [1 / n_clusters] * n_clusters) + 1
    mean = lowest + rng.uniform(0.05, 0.6, size=n_clusters)   # mean sits above the lowest
    lines = [
        "AutoDock4 log file (demo)",
        "",
        "________",
        "CLUSTERING HISTOGRAM",
        "__________________",
        "______________________________________________________________________",
        "Cluster Rank | Lowest Energy | Run | Mean Energy | # in Cluster",
        "_____________|_______________|_____|_____________|________________",
    ]
    run = int(rng.integers(1, total_runs))
    for rank in range(n_clusters):
        lines.append(
            f"     {rank + 1}       |     {lowest[rank]:7.2f}    |  {run:3d} |"
            f"    {mean[rank]:7.2f}    |   {sizes[rank]:3d} |"
        )
    lines += [
        "____________|_______________|_____|_____________|________________",
        "",
        "Successful Completion",
    ]
    return "\n".join(lines) + "\n"


CAMPAIGN_LIGANDS = [
    ("aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
    ("ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ("naproxen", "COc1ccc2cc(ccc2c1)C(C)C(=O)O"),
    ("paracetamol", "CC(=O)Nc1ccc(O)cc1"),
    ("caffeine", "Cn1cnc2c1c(=O)n(C)c(=O)n2C"),
    ("theophylline", "Cn1c(=O)c2[nH]cnc2n(C)c1=O"),
    ("metformin", "CN(C)C(=N)N=C(N)N"),
    ("benzocaine", "CCOC(=O)c1ccc(N)cc1"),
    ("propranolol", "CC(C)NCC(O)COc1cccc2ccccc12"),
    ("nicotine", "CN1CCCC1c1cccnc1"),
    ("atenolol", "CC(C)NCC(O)COc1ccc(CC(N)=O)cc1"),
    ("lidocaine", "CCN(CC)CC(=O)Nc1c(C)cccc1C"),
    ("coumarin", "O=C1Oc2ccccc2C=C1"),
    ("vanillin", "COc1cc(C=O)ccc1O"),
]

DOCKING_DIR = WORK / "docking_run"     # <- replace with your campaign folder
for i, (name, _) in enumerate(CAMPAIGN_LIGANDS, start=1):
    folder = DOCKING_DIR / f"{i:02d}_{name}"
    folder.mkdir(parents=True, exist_ok=True)
    if name == "benzocaine":
        continue                          # no r.dlg written -> MISSING_LOG
    if name == "nicotine":
        (folder / "r.dlg").write_text(
            "AutoDock4 log file (demo)\n\n... run interrupted before clustering\n",
            encoding="utf-8",
        )
        (folder / "r.glg").write_text("autogrid4 log (demo)\n", encoding="utf-8")
        continue                          # no cluster table -> PARSE_ERROR
    (folder / "r.dlg").write_text(make_dlg(seed=100 + i), encoding="utf-8")
    (folder / "r.glg").write_text("autogrid4 log (demo)\n", encoding="utf-8")

print(f"demo campaign: {DOCKING_DIR}")
print(f"{len(list(DOCKING_DIR.iterdir()))} ligand folders "
      f"(12 OK, 1 missing log, 1 interrupted)")

In [ ]:
from pharmaml.docking import parse_dlg_cluster_table

clusters = parse_dlg_cluster_table(DOCKING_DIR / "01_aspirin" / "r.dlg")
pd.DataFrame([asdict(c) for c in clusters])

In [ ]:
from pharmaml.docking import parse_dlg_summary

summary_one = parse_dlg_summary(DOCKING_DIR / "01_aspirin" / "r.dlg")
for key, value in summary_one.items():
    print(f"{key:22s}: {value}")
print("\n'top_' = lowest energy overall; 'best_' = most populated cluster")

In [ ]:
from pharmaml.docking import check_docking_completion

audit = check_docking_completion(DOCKING_DIR)
print(audit["status"].value_counts().to_dict(), "\n")
audit

In [ ]:
from pharmaml.docking import scan_docking_directory

summary = scan_docking_directory(DOCKING_DIR)
summary.round(2)

In [ ]:
import warnings

from pharmaml.docking import batch_run_directories

# IPython resets warning filters per cell, so scope the fork() noise here.
warnings.filterwarnings("ignore", message=".*use of fork.*")

# Idempotent campaign re-run: folders whose outputs (.glg/.dlg) already
# exist are skipped, per-folder failures are collected (benzocaine has no
# files at all), and the interrupted run is *not* re-docked because its
# .dlg exists -- which is exactly why the completion audit above exists
# as a separate check.
for r in batch_run_directories(DOCKING_DIR, workers=2):
    status = "ok" if r.ok else f"FAILED: {r.error}"
    print(f"[batch] {r.directory.name:16s} -> {status}")

In [ ]:
from pharmaml.docking import collect_files, reindex_by_number

# Campaign housekeeping (upgrades extaacer.py + Prettier_extracted_dlgs.py):
# gather every r.dlg of the tree into one folder, renamed after its
# ligand folder (with _1/_2 dedup suffixes) so results never overwrite…
collected = collect_files(DOCKING_DIR, filename="r.dlg")
print(collected[["folder", "destination", "copied"]].to_string(index=False))

# …then reindex the campaign summary onto the full ligand-number range
# so missing ligands surface as visible NaN rows instead of silently
# absent ones (00 here is the numbering gap by construction).
aligned = reindex_by_number(summary, column="folder", start=1, stop=14)
present = aligned["folder"].notna().sum()
print(f"\nsummary rows reindexed onto 1..14: {len(aligned)} "
      f"({int(present)} present, {len(aligned) - int(present)} missing)")

## 7 · Parse Vina output — pose tables

`parse_vina_output` reads either the stdout mode table or the
`REMARK VINA RESULT` lines of an output PDBQT, and returns typed poses.


In [ ]:
from pharmaml.docking import parse_vina_output

# REMARK lines as written into a real Vina output PDBQT.
vina_pdbqt = "\n".join([
    "REMARK VINA RESULT:   -10.3      0.000      0.000",
    "REMARK VINA RESULT:    -9.6      1.842      2.301",
    "REMARK VINA RESULT:    -9.4      2.105      2.777",
    "REMARK VINA RESULT:    -8.9      2.733      3.418",
    "REMARK VINA RESULT:    -8.1      4.006      5.114",
])

poses = parse_vina_output(vina_pdbqt)
pd.DataFrame([asdict(p) for p in poses])

## 8 · Rescoring — ligand efficiency + consensus

Raw binding energy favours big molecules; `ligand_efficiency` normalises by
heavy-atom count (Hopkins 2004). `consensus_rescore` then min-max normalises
several evidence columns — energy (lower better), cluster population (higher
better), ligand efficiency — into one 0–1 score.  Heavy-atom counts are
computed from the actual SMILES with RDKit.


In [ ]:
from pharmaml.docking import ligand_efficiency, ligand_efficiency_series

# Rank only the ligands that actually completed.
ok = summary[summary["status"] == "OK"].copy()
ok["ligand"] = ok["folder"].str.split("_").str[1]

# Heavy-atom counts from the actual SMILES (RDKit).
from rdkit import Chem

smiles_map = dict(CAMPAIGN_LIGANDS)
ok["num_heavy_atoms"] = [
    Chem.MolFromSmiles(smiles_map[name]).GetNumHeavyAtoms() for name in ok["ligand"]
]

ok["ligand_efficiency"] = ligand_efficiency_series(ok)
ok[["folder", "best_lowest_energy", "num_heavy_atoms", "ligand_efficiency"]].round(3)

In [ ]:
from pharmaml.docking import consensus_rescore

# Energy (lower better) + cluster population (higher better) + LE (higher
# better) -> one 0-1 consensus; direction defaults handle all three.
ok["consensus"] = consensus_rescore(
    ok, columns=["best_lowest_energy", "best_num_in_cluster", "ligand_efficiency"]
)

ranked = ok.sort_values("consensus", ascending=False).reset_index(drop=True)
ranked[["folder", "best_lowest_energy", "best_num_in_cluster",
        "ligand_efficiency", "consensus"]].round(3).head(8)

## 9 · Visualization — energy distribution & top ligands

Two figures the legacy archive never drew: the campaign-wide energy
histogram (with median and best ligand marked) and the ranked top-ligand
bars (strongest binding on top).


In [ ]:
import warnings

from pharmaml.viz import plot_energy_distribution

# Version-compatibility noise from matplotlib/pyparsing in this env.
warnings.filterwarnings("ignore", category=DeprecationWarning)

fig_energy = plot_energy_distribution(
    ok, title="Campaign binding energies (best cluster per ligand)"
)

In [ ]:
from pharmaml.viz import plot_top_ligands

fig_top = plot_top_ligands(ok, top_n=10, title="Top 10 ligands by binding energy")

## 10 · Publish — the one-click visual bundle

`save_visual_bundle` renders the figures and tables into a directory **and**
a self-contained `report.html` (figures embedded as base64 — no external
assets).  `export_table` writes any table as csv/tsv/markdown/json/html.


In [ ]:
from pharmaml.viz import save_visual_bundle

campaign_metrics = {
    "ligands docked": int(len(ok)),
    "failed folders": int((summary["status"] != "OK").sum()),
    "best energy (kcal/mol)": float(ok["best_lowest_energy"].min()),
    "best ligand": str(ok.loc[ok["best_lowest_energy"].idxmin(), "folder"]),
    "median energy (kcal/mol)": float(ok["best_lowest_energy"].median()),
    "mean ligand efficiency": round(float(ok["ligand_efficiency"].mean()), 3),
}

BUNDLE = WORK / "docking_reports"
artifacts = save_visual_bundle(
    figures={"energy_distribution": fig_energy, "top_ligands": fig_top},
    tables={
        "docking_summary": ok[["folder", "n_clusters", "best_lowest_energy",
                               "best_mean_energy", "best_num_in_cluster",
                               "num_heavy_atoms", "ligand_efficiency",
                               "consensus"]],
        "completion_audit": audit,
        "vina_poses": pd.DataFrame([asdict(p) for p in poses]),
    },
    metrics=campaign_metrics,
    out_dir=BUNDLE,
    title="Docking campaign report",
    subtitle=f"{len(ok)} ligands · AutoDock4 demo campaign",
    formats=("png", "svg"),
    close=False,   # keep figures open so they also render above
)
pd.DataFrame({"artifact": [p.name for p in artifacts],
              "kind": [p.suffix.lstrip(".") for p in artifacts]})

In [ ]:
from pharmaml.viz import export_table

exports = [
    export_table(ok, BUNDLE / "docking_summary.md"),
    export_table(ranked, BUNDLE / "consensus_ranking.csv"),
    export_table(audit, BUNDLE / "completion_audit.json"),
]
report_html = BUNDLE / "report.html"
print(f"{report_html}  ({report_html.stat().st_size / 1024:.0f} KB, zero external assets)")
[p.name for p in exports]

## 11 · Recap

| Stage | One-liner |
|---|---|
| Import ligand CSV | `load_ligand_csv(ligands.csv)` → validated `LigandLibrary` |
| Batch ligand prep | `prepare_ligand_library(library, out_dir)` → folder-per-ligand tree |
| Import & prep protein | `prepare_protein(protein.pdb, r.pdbqt)` → cleaned, typed, charged receptor |
| Distribute receptor | `distribute_receptor(r.pdbqt, campaign_root)` |
| Single ligand | `smiles_to_pdbqt(smiles, out_dir, name=...)` |
| Grid box | `compute_grid_box(reference_ligand.pdbqt, padding=4)` |
| GPF file | `write_gpf(ligand.pdbqt, box=box)` |
| DPF file | `write_dpf(ligand.pdbqt, ga_run=100)` |
| Torsion QA | `scan_torsion_consistency(ligand_dir)` → OK / MISMATCH |
| Run AutoDock4 | `run_autodock4(ligand_dir)` / `batch_run_directories(base)` |
| Run Vina | `run_vina(receptor, ligand, out, center, size)` |
| One ligand | `parse_dlg_summary(r.dlg)` |
| Audit campaign | `check_docking_completion(docking_run)` |
| Scan campaign | `scan_docking_directory(docking_run)` |
| Vina poses | `parse_vina_output(out.pdbqt)` |
| Rescore | `ligand_efficiency_series(frame)` + `consensus_rescore(frame, cols)` |
| Visualize | `plot_energy_distribution(summary)` / `plot_top_ligands(summary)` |
| Publish | `save_visual_bundle(figures, tables, metrics, "reports")` |

CLI equivalents:

```bash
pharmaml docking prep-ligands --ligands ligands.csv --out ligand_folders/
pharmaml docking prep-protein --protein protein.pdb --out r.pdbqt
pharmaml docking scan      --directory docking_run/ --out summary.csv
pharmaml docking check     --directory docking_run/
```
